# 🚲 Cycle Tracking Pipeline (Colab version)

Run each cell top to bottom, in order. Where a cell asks you to fill something in, do that before running the next one.

**What this notebook does:** downloads your registered-cycle photos + sighting photos from Drive, builds a visual fingerprint per cycle with CLIP, matches sightings to registered cycles, and lets you search "where was CYC-001 last seen?".

## 1. Install dependencies

In [ ]:
!pip install -q gdown open_clip_torch pyarrow
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
done


In [ ]:
import os, re, csv, time
import numpy as np
import pandas as pd
from PIL import Image
import torch
import open_clip
import gdown
from google.colab import files

os.makedirs("data/registrations", exist_ok=True)
os.makedirs("data/sightings/Day1", exist_ok=True)
print("folders ready")

folders ready


In [ ]:
uploaded = files.upload()  # pick your registrations.csv here
csv_name = list(uploaded.keys())[0]
os.rename(csv_name, "registrations.csv")
print("saved as registrations.csv")

Saving registerations.csv to registerations.csv
saved as registrations.csv


In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io

drive_service = build('drive', 'v3')

def download_drive_file_auth(file_id, dest_path):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.FileIO(dest_path, 'wb')
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
    fh.close()

In [ ]:
PHOTO_COL_KEYWORDS = ["photo", "upload", "image", "picture"]
DRIVE_ID_RE = re.compile(r"[-\w]{25,}")

def extract_file_ids(cell_value):
    if not cell_value:
        return []
    ids = []
    for part in cell_value.split(","):
        m = DRIVE_ID_RE.search(part.strip())
        if m:
            ids.append(m.group(0))
    return ids

def find_photo_column(fieldnames):
    for col in fieldnames:
        if any(k in col.lower() for k in PHOTO_COL_KEYWORDS):
            return col
    return None

with open("registrations.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    photo_col = find_photo_column(reader.fieldnames)
    print(f"Using photo column: '{photo_col}'")
    rows = list(reader)

os.makedirs("data/registrations", exist_ok=True)
index_rows = []
for i, row in enumerate(rows):
    cycle_id = f"CYC-{i+1:03d}"
    cycle_dir = os.path.join("data/registrations", cycle_id)
    os.makedirs(cycle_dir, exist_ok=True)
    ids = extract_file_ids(row.get(photo_col, ""))
    downloaded = 0
    for j, fid in enumerate(ids):
        dest = os.path.join(cycle_dir, f"photo_{j+1}.jpg")
        try:
            gdown.download(id=fid, output=dest, quiet=True)
            downloaded += 1
        except Exception as e:
            print(f"[ERROR] {cycle_id} photo {j+1}: {e}")
    print(f"  {cycle_id}: {downloaded}/{len(ids)} photos downloaded")
    index_rows.append({"cycle_id": cycle_id, **row, "num_photos_downloaded": downloaded})

pd.DataFrame(index_rows).to_csv("data/registrations_index.csv", index=False)
print("done")

Using photo column: 'Cycle Photos (Upload 3 images)'
  CYC-001: 3/3 photos downloaded
  CYC-002: 3/3 photos downloaded
  CYC-003: 3/3 photos downloaded
  CYC-004: 3/3 photos downloaded
  CYC-005: 3/3 photos downloaded
  CYC-006: 2/2 photos downloaded
  CYC-007: 3/3 photos downloaded
  CYC-008: 3/3 photos downloaded
  CYC-009: 3/3 photos downloaded
  CYC-010: 3/3 photos downloaded
done


In [ ]:
SIGHTINGS_FOLDER_URL = "https://drive.google.com/drive/folders/1RWUAtrvtdkNOXsj9JFrvtTM8Ol-lFp7k"  # <-- your folder link
folder_id = re.search(r"folders/([-\w]+)", SIGHTINGS_FOLDER_URL).group(1)

os.makedirs("data/sightings/Day1", exist_ok=True)

page_token = None
files_list = []
while True:
    resp = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="nextPageToken, files(id, name)",
        pageToken=page_token,
    ).execute()
    files_list.extend(resp.get("files", []))
    page_token = resp.get("nextPageToken")
    if not page_token:
        break

print(f"found {len(files_list)} files in folder")

downloaded = 0
for f in files_list:
    dest = os.path.join("data/sightings/Day1", f["name"])
    try:
        download_drive_file_auth(f["id"], dest)
        downloaded += 1
    except Exception as e:
        print(f"[ERROR] {f['name']}: {e}")

print("\ndownloaded:", downloaded, "/", len(files_list), "files")

found 10 files in folder

downloaded: 10 / 10 files


In [ ]:
SIGHTINGS_ROOT = "data/sightings"
FNAME_RE = re.compile(r"^(?P<location>[A-Za-z]+)_(?P<time>\d{3,4})_(?P<collector>[A-Za-z]+)_(?P<seq>\d+)\.(jpg|jpeg|png)$", re.IGNORECASE)

def parse_filename(fname):
    m = FNAME_RE.match(fname)
    if not m:
        return None
    time_str = m.group("time").zfill(4)
    return {
        "location": m.group("location"),
        "time": f"{time_str[:2]}:{time_str[2:]}",
        "collector": m.group("collector"),
        "seq": int(m.group("seq")),
    }

records, skipped = [], []
for day_folder in sorted(os.listdir(SIGHTINGS_ROOT)):
    day_path = os.path.join(SIGHTINGS_ROOT, day_folder)
    if not os.path.isdir(day_path):
        continue
    for fname in sorted(os.listdir(day_path)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        parsed = parse_filename(fname)
        if parsed is None:
            skipped.append(fname)
            continue
        records.append({
            "sighting_id": f"{day_folder}_{fname}",
            "day": day_folder,
            "image_path": os.path.join(day_path, fname),
            **parsed,
        })

sightings_df = pd.DataFrame(records)
sightings_df.to_csv("data/sightings_index.csv", index=False)
print(f"Parsed {len(sightings_df)} sighting photos -> data/sightings_index.csv")
if skipped:
    print(f"\n[WARN] {len(skipped)} file(s) skipped (bad filename):")
    for f in skipped:
        print("   -", f)

Parsed 10 sighting photos -> data/sightings_index.csv


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONF_THRESHOLD = 0.75  # tune this after eyeballing real confidence scores below

print(f"Loading CLIP on {DEVICE} ...")
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k", device=DEVICE)
model.eval()

def embed_image(path):
    img = Image.open(path).convert("RGB")
    tensor = preprocess(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]

# --- registry embeddings ---
cycle_ids, embeddings = [], []
for cycle_id in sorted(os.listdir("data/registrations")):
    cdir = os.path.join("data/registrations", cycle_id)
    if not os.path.isdir(cdir):
        continue
    vecs = []
    for fname in sorted(os.listdir(cdir)):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            try:
                vecs.append(embed_image(os.path.join(cdir, fname)))
            except Exception as e:
                print(f"[WARN] {cdir}/{fname}: {e}")
    if vecs:
        avg = np.mean(vecs, axis=0)
        avg = avg / np.linalg.norm(avg)
        cycle_ids.append(cycle_id)
        embeddings.append(avg)

registry_matrix = np.array(embeddings, dtype=np.float32)
print(f"{len(cycle_ids)} cycles embedded")

# --- match sightings ---
sightings_df = pd.read_csv("data/sightings_index.csv")
matched_ids, confidences, gaps = [], [], []
all_sims = []   # <-- NEW: store every sighting's similarity to EVERY registered cycle

for _, row in sightings_df.iterrows():
    try:
        vec = embed_image(row["image_path"])
        sims = registry_matrix @ vec
        all_sims.append(sims)   # <-- NEW
        order = np.argsort(-sims)
        best_score = float(sims[order[0]])
        second_score = float(sims[order[1]]) if len(order) > 1 else 0.0
        matched_ids.append(cycle_ids[order[0]] if best_score >= CONF_THRESHOLD else None)
        confidences.append(round(best_score, 4))
        gaps.append(round(best_score - second_score, 4))
    except Exception as e:
        print(f"[WARN] {row['image_path']}: {e}")
        all_sims.append(np.zeros(len(cycle_ids)))   # <-- NEW
        matched_ids.append(None); confidences.append(None); gaps.append(None)

all_sims = np.array(all_sims)   # <-- NEW: shape (n_sightings, n_cycles)

sightings_df["matched_cycle_id"] = matched_ids
sightings_df["confidence"] = confidences
sightings_df["top2_gap"] = gaps

sightings_df.to_parquet("data/sightings_matched.parquet", index=False)
sightings_df.to_csv("data/sightings_matched.csv", index=False)

n_matched = sightings_df["matched_cycle_id"].notna().sum()
print(f"\n{n_matched}/{len(sightings_df)} sightings matched at >= {CONF_THRESHOLD} confidence")
sightings_df.sort_values("confidence", ascending=False)

Loading CLIP on cuda ...


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

10 cycles embedded

1/10 sightings matched at >= 0.75 confidence


,sighting_id,day,image_path,location,time,collector,seq,matched_cycle_id,confidence,top2_gap
4,Day1_KAMENG_0230_SV_01.jpg,Day1,data/sightings/Day1/KAMENG_0230_SV_01.jpg,KAMENG,02:30,SV,1,CYC-001,0.7552,0.0164
9,Day1_LOHIT_0340_SB_04.jpg,Day1,data/sightings/Day1/LOHIT_0340_SB_04.jpg,LOHIT,03:40,SB,4,None,0.7401,0.0449
7,Day1_LOHIT_0330_SJ_02.jpg,Day1,data/sightings/Day1/LOHIT_0330_SJ_02.jpg,LOHIT,03:30,SJ,2,None,0.7339,0.0513
1,Day1_BRAHMAPUTRA_0135_SJ_01.jpg,Day1,data/sightings/Day1/BRAHMAPUTRA_0135_SJ_01.jpg,BRAHMAPUTRA,01:35,SJ,1,None,0.7044,0.0070
5,Day1_KAPILI_1235_SB_01.jpg,Day1,data/sightings/Day1/KAPILI_1235_SB_01.jpg,KAPILI,12:35,SB,1,None,0.6982,0.0116
6,Day1_LOHIT_0100_SB_01.jpg,Day1,data/sightings/Day1/LOHIT_0100_SB_01.jpg,LOHIT,01:00,SB,1,None,0.6549,0.0287
8,Day1_LOHIT_0330_SV_03.jpg,Day1,data/sightings/Day1/LOHIT_0330_SV_03.jpg,LOHIT,03:30,SV,3,None,0.6372,0.0042
2,Day1_CONFERENCE_0200_SV_01.jpg,Day1,data/sightings/Day1/CONFERENCE_0200_SV_01.jpg,CONFERENCE,02:00,SV,1,None,0.6136,0.0040
0,Day1_ADMIN_0935_SJ_01.jpg,Day1,data/sightings/Day1/ADMIN_0935_SJ_01.jpg,ADMIN,09:35,SJ,1,None,0.5485,0.0534
3,Day1_DIHING_1105_SV_01.jpg,Day1,data/sightings/Day1/DIHING_1105_SV_01.jpg,DIHING,11:05,SV,1,None,0.5394,0.0042


In [ ]:
SEARCH_ID = "CYC-008"  # <-- change this
TOP_N = 5              # <-- kitni locations dikhani hain shortlist mein

if SEARCH_ID not in cycle_ids:
    print(f"{SEARCH_ID} registered cycles mein nahi mili.")
else:
    col = cycle_ids.index(SEARCH_ID)
    ranked = sightings_df.copy()
    ranked["match_score"] = all_sims[:, col]
    ranked = ranked.sort_values("match_score", ascending=False).head(TOP_N)

    print(f"{SEARCH_ID} ke liye top {len(ranked)} suggested locations (visual similarity se ranked):\n")
    for _, row in ranked.iterrows():
        tag = "CONFIDENT" if row["match_score"] >= CONF_THRESHOLD else "possible"
        print(f"[{tag}] {row['location']} | {row['day']} {row['time']} | score {row['match_score']:.4f}")

CYC-008 ke liye top 5 suggested locations (visual similarity se ranked):

[possible] KAMENG | Day1 02:30 | score 0.7388
[possible] BRAHMAPUTRA | Day1 01:35 | score 0.6975
[possible] KAPILI | Day1 12:35 | score 0.6865
[possible] LOHIT | Day1 03:30 | score 0.6826
[possible] LOHIT | Day1 03:40 | score 0.6629


In [ ]:
# --- Export for the Cycle Search app ---
export_df = sightings_df[["sighting_id", "day", "location", "time", "image_path"]].copy()
for i, cid in enumerate(cycle_ids):
    export_df[cid] = all_sims[:, i]

export_df.to_csv("data/similarity_matrix.csv", index=False)
print("saved data/similarity_matrix.csv with columns:", list(export_df.columns))

import shutil
shutil.make_archive("cycle_search_data", "zip", "data")
files.download("cycle_search_data.zip")

saved data/similarity_matrix.csv with columns: ['sighting_id', 'day', 'location', 'time', 'image_path', 'CYC-001', 'CYC-002', 'CYC-003', 'CYC-004', 'CYC-005', 'CYC-006', 'CYC-007', 'CYC-008', 'CYC-009', 'CYC-010']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("data/sightings_matched.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>